# 03 PostgreSQL / pgvector 体检

**测什么**: `.env` 里的库配置能否连上、pgvector 扩展是否可用、业务表与
checkpointer/store 表是否就绪、各表数据量。

**判定**: 连不上为 FAIL(附修复提示)。库不通时后端会降级 InMemory, 图仍能跑,
但法条检索、案例检索、长期记忆、审计全部失效。

**前置**: 无(库不通也必须能得出明确结论)。


In [ ]:
import asyncio, os, sys
from pathlib import Path

for cand in (Path.cwd(), *Path.cwd().parents):
    if (cand / "nbkit.py").is_file():
        NB_DIR = cand
        break
    if (cand / "tests_ipynb" / "nbkit.py").is_file():
        NB_DIR = cand / "tests_ipynb"
        break
else:
    raise RuntimeError("未找到 nbkit.py")

sys.path.insert(0, str(NB_DIR))

from nbkit import Checks, bootstrap

ROOT = bootstrap()
checks = Checks("03 PostgreSQL 体检")

print("解释器  :", sys.executable)
print("仓库根  :", ROOT)
print("HF_HOME :", os.getenv("HF_HOME", "(未设置)"))


In [ ]:
from lawApp_LangGraph.config import settings as s
from lawApp_LangGraph.db import build_dsn
from nbkit import mask

dsn = build_dsn()
print("目标库  :", dsn.rsplit("@", 1)[-1])
print("用户    :", s.db_user)
print("密码    :", mask(s.db_password or ""))
print("backend :", s.checkpoint_backend)
if s.database_url:
    print("DATABASE_URL 覆盖生效:", mask(s.database_url))

## 1. 建连(3 秒超时, 不阻塞)

In [ ]:
import psycopg

dsn_plain = dsn.replace("postgresql+psycopg", "postgresql")
conn = None
try:
    conn = psycopg.connect(dsn_plain, connect_timeout=3)
except Exception as e:
    checks.fail(
        "PostgreSQL 建连",
        f"{type(e).__name__}: {str(e)[:160]} → 修 .env 的 DB_PASSWORD 或 DATABASE_URL, "
        "或把 CHECKPOINT_BACKEND 设为 memory 走无库模式",
    )
else:
    checks.ok("PostgreSQL 建连", f"server={conn.info.server_version}")

## 2. pgvector 扩展

In [ ]:
if conn is not None:
    with conn.cursor() as cur:
        cur.execute("SELECT extversion FROM pg_extension WHERE extname = 'vector'")
        row = cur.fetchone()
    if row:
        checks.ok("pgvector 扩展可用", f"version={row[0]}")
    else:
        checks.fail("pgvector 扩展可用", "未安装 → 需在库内 CREATE EXTENSION vector(需超管或已装扩展)")
else:
    checks.skip("pgvector 扩展可用", "未建连")

## 3. 业务表与数据量

In [ ]:
BIZ = ("law_vector", "law_cases", "sessions", "audit", "feedback")

if conn is not None:
    with conn.cursor() as cur:
        cur.execute("SELECT tablename FROM pg_tables WHERE schemaname = 'public'")
        tables = {r[0] for r in cur.fetchall()}
        cur.execute("SELECT tablename FROM pg_tables WHERE schemaname = 'public'")
    for t in BIZ:
        if t not in tables:
            checks.fail(f"业务表 {t}", "不存在 → 由 db.get_pool() 首次调用时自动建表(见下节)")
            continue
        with conn.cursor() as cur:
            cur.execute(f"SELECT count(*) FROM {t}")
            n = cur.fetchone()[0]
        checks.expect(
            n > 0,
            f"业务表 {t} 有数据",
            ok_detail=f"{n} 行",
            fail_detail="0 行 → 表已建但未灌数据(法条/案例需入库脚本, 该脚本当前不存在)",
        )
    others = sorted(t for t in tables if t not in BIZ)
    checks.skip("其余表(checkpointer/store)", ", ".join(others) if others else "无")
else:
    for t in BIZ:
        checks.skip(f"业务表 {t}", "未建连")

## 4. 走一遍真实代码路径 `db.get_pool()`(自动建表)

In [ ]:
if conn is not None:
    conn.close()
    from lawApp_LangGraph.db import close_pool, get_pool

    try:
        pool = await asyncio.wait_for(get_pool(), 30)
        async with pool.connection() as c2:
            cur = await c2.execute("SELECT count(*) FROM law_vector")
            n_law = (await cur.fetchone())[0]
            cur = await c2.execute("SELECT count(*) FROM law_cases")
            n_case = (await cur.fetchone())[0]
        checks.ok("db.get_pool() 就绪", f"law_vector={n_law} 行 | law_cases={n_case} 行")
    except Exception as e:
        checks.fail("db.get_pool() 就绪", f"{type(e).__name__}: {str(e)[:160]}")
    finally:
        await close_pool()
else:
    checks.skip("db.get_pool() 就绪", "未建连")

## 汇总

In [ ]:
print(checks.report())